<a href="https://colab.research.google.com/github/IGol22/SFML/blob/main/%D0%9A%D1%83%D1%80%D1%81%D0%BE%D0%B2%D0%B0%D1%8F/%D0%A0%D0%B5%D0%B3%D1%80%D0%B5%D1%81%D1%81%D0%B8%D1%8FSI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q catboost

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              AdaBoostRegressor, HistGradientBoostingRegressor, StackingRegressor)
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Автоматическая подгрузка с GitHub
file_name = "dataset_classic_ML.xlsx"
if not os.path.exists(file_name):
    raw_url = "https://raw.githubusercontent.com/IGol22/SFML/main/Курсовая/dataset_classic_ML.xlsx"
    !wget -q -O {file_name} "{raw_url}"

df = pd.read_excel(file_name)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.4 MB/s eta 0:00:00


,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,...,0,0,0,0,0,0,0,0,3,0
1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,...,0,0,0,0,0,0,0,0,3,0
2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,...,0,0,0,0,0,0,0,0,3,0
3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,398.679,...,0,0,0,0,0,0,0,0,4,0
4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,466.713,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# 1. Задаем таргет SI и исключаем утечки
target_col = 'SI'
targets_to_drop = ['IC50, mM', 'CC50, mM', 'SI']

X = df.drop(columns=targets_to_drop, errors='ignore').copy()

# Логарифмируем SI по основанию 10 для нормализации масштаба
y = np.log10(df[target_col])

# 2. Feature Engineering
if 'MolLogP' in X.columns and 'MolWt' in X.columns:
    X['MolLogP_x_MolWt'] = X['MolLogP'] * X['MolWt']

poly_cols = [c for c in ['MolLogP', 'MolWt'] if c in X.columns]
if poly_cols:
    poly = PolynomialFeatures(degree=2, include_bias=False)
    poly_feats = poly.fit_transform(X[poly_cols])
    poly_df = pd.DataFrame(poly_feats, columns=poly.get_feature_names_out(poly_cols), index=X.index)
    for col in poly_df.columns:
        if col not in X.columns:
            X[col] = poly_df[col]

if 'MolLogP' in X.columns:
    X['MolLogP_gt_3'] = (X['MolLogP'] > 3).astype(int)

# 3. Заполнение пропусков
if X.isnull().values.any():
    imputer = SimpleImputer(strategy='median')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(f"Размерность матрицы X: {X.shape}, Длина y: {len(y)}")

Размерность матрицы X: (1001, 215), Длина y: 1001


In [ ]:
# Разбиение
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Отбор признаков и масштабирование
vt = VarianceThreshold(threshold=0.01)
X_train_sel = vt.fit_transform(X_train)
X_test_sel = vt.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

# Модели
models = {
    'KNN': KNeighborsRegressor(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'CatBoost': CatBoostRegressor(random_state=42, verbose=0),
    'Stacking': StackingRegressor(
        estimators=[
            ('rf', RandomForestRegressor(random_state=42)),
            ('gb', GradientBoostingRegressor(random_state=42)),
            ('xgb', XGBRegressor(random_state=42))
        ],
        final_estimator=LinearRegression()
    )
}

# Цикл обучения
results = []
for name, model in models.items():
    tr_x = X_train_scaled if name in ['KNN'] else X_train_sel
    te_x = X_test_scaled if name in ['KNN'] else X_test_sel

    model.fit(tr_x, y_train)
    y_pred = model.predict(te_x)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({'Model': name, 'MSE': mse, 'RMSE': rmse, 'R2': r2})

res_df = pd.DataFrame(results).sort_values(by='R2', ascending=False).round(3)
res_df

,Model,MSE,RMSE,R2
7,Stacking,0.444,0.666,0.272
1,Random Forest,0.445,0.667,0.270
0,KNN,0.480,0.693,0.214
6,CatBoost,0.480,0.693,0.214
3,HistGradientBoosting,0.483,0.695,0.209
2,Gradient Boosting,0.491,0.701,0.195
4,AdaBoost,0.530,0.728,0.131
5,XGBoost,0.538,0.734,0.118


## Выводы и рекомендации (Регрессия SI)

### Сравнение моделей
* **Лидеры:** Лучше всех сработали **Stacking** (R² = 0.272, RMSE = 0.666) и **Random Forest** (R² = 0.270, RMSE = 0.667). По сути, они выдали одинаковый результат.
* **Сложность показателя:** Точность тут ниже, чем для IC50 и CC50. Это нормально, ведь SI — это отношение двух величин (CC50 / IC50). Ошибки измерений из обоих тестов накладываются друг на друга, и предсказывать SI напрямую сложнее всего.
* **Предобработка:** Логарифмирование (log10) помогло убрать дикий разброс значений и нормально обучить модели.

---

### Рекомендации
* **Выбор модели:** Для работы стоит выбирать Stacking или Random Forest.
* **Практический подход:** Чтобы точнее оценивать молекулы, надежнее смотреть на отдельные прогнозы для IC50 и CC50, а не только на один лишь SI.